In [1]:
import json
import pandas as pd

conversations = []
with open("../data/processed/amazon_conversations_clean.jsonl", encoding="utf-8") as f:
    for line in f:
        conversations.append(json.loads(line))

print("Total clean conversations:", len(conversations))

Total clean conversations: 60669


In [2]:
first_customer_msgs = []
for conv in conversations:
    first_turn = next((t for t in conv["turns"] if t["role"] == "CUSTOMER"), None)
    if first_turn and len(first_turn["cleaned_text"].strip()) > 0:
        first_customer_msgs.append({
            "conversation_id": conv["conversation_id"],
            "text": first_turn["cleaned_text"]
        })

df_msgs = pd.DataFrame(first_customer_msgs)
print("Total first-customer-messages:", len(df_msgs))
df_msgs.head(10)

Total first-customer-messages: 60669


,conversation_id,text
0,1000123,I just inadvertently bought a Kindle book on m...
1,1000125,Please upload south movies in hindi audio
2,1000127,watching Loveless on Amazon prime cuz of your ...
3,1000130,sign up page appears broken. Cannot sign up. H...
4,1000133,Amazon now sell duplicate product. I bought Bi...
5,1000142,Love how these deliveries have been left outsi...
6,1000154,Is your company bound to destroy people's diwa...
7,1000164,This is after package was supposedly out for d...
8,1000195,Looks like it’s time to invest in Best Buy’s G...
9,1000201,. Does Kindle support Kannada (Unicode renderi...


In [3]:
from collections import Counter
import re

# Simple word frequency (stopwords ke bina rough idea ke liye)
all_words = []
for text in df_msgs["text"]:
    words = re.findall(r'\b[a-z]{4,}\b', text.lower())
    all_words.extend(words)

word_freq = Counter(all_words)
word_freq.most_common(40)

[('amazon', 13352),
 ('delivery', 10902),
 ('order', 10861),
 ('this', 10530),
 ('have', 9467),
 ('prime', 8743),
 ('with', 8580),
 ('your', 7812),
 ('that', 7774),
 ('from', 7322),
 ('delivered', 6051),
 ('service', 5925),
 ('what', 5717),
 ('when', 5710),
 ('been', 5104),
 ('they', 5066),
 ('customer', 5045),
 ('package', 4730),
 ('ordered', 4708),
 ('today', 4338),
 ('just', 4304),
 ('help', 4303),
 ('please', 4218),
 ('time', 4188),
 ('days', 4049),
 ('still', 4025),
 ('will', 3405),
 ('product', 3260),
 ('account', 3013),
 ('there', 2904),
 ('received', 2734),
 ('shipping', 2641),
 ('even', 2585),
 ('deliver', 2521),
 ('refund', 2471),
 ('item', 2460),
 ('need', 2399),
 ('after', 2242),
 ('like', 2206),
 ('back', 2157)]

In [4]:
df_msgs["word_count"] = df_msgs["text"].apply(lambda x: len(x.split()))
df_msgs["word_count"].describe()

count    60669.000000
mean        21.234584
std          9.687581
min          1.000000
25%         15.000000
50%         20.000000
75%         25.000000
max         61.000000
Name: word_count, dtype: float64

In [5]:
import re

# In words ke basis pe, rough candidate intent buckets bana rahe hai
# (final taxonomy nahi hai abhi — sirf hypothesis test karne ke liye)
candidate_buckets = {
    "delivery_status": [r"\bdeliver", r"\bpackage\b", r"\btrack", r"\barriv", r"\bshipp", r"\bwhere is\b"],
    "refund_return": [r"\brefund", r"\breturn", r"\bmoney back\b", r"\bexchange\b"],
    "order_issue": [r"\border\b", r"\bordered\b", r"\bcancel", r"\bwrong item\b"],
    "account_access": [r"\baccount\b", r"\bpassword\b", r"\blogin\b", r"\block", r"\baccess\b"],
    "payment_billing": [r"\bcharge", r"\bpayment\b", r"\bbill", r"\bcard\b", r"\bmoney\b"],
    "prime_membership": [r"\bprime\b", r"\bmembership\b", r"\bsubscription\b"],
    "product_issue": [r"\bdamaged\b", r"\bbroken\b", r"\bdefective\b", r"\bnot working\b", r"\bfaulty\b"],
    "customer_service_complaint": [r"\bcustomer service\b", r"\brude\b", r"\bworst\b", r"\bterrible\b", r"\bdisappointed\b"],
    "human_request": [r"\bhuman\b", r"\breal person\b", r"\bspeak to\b", r"\bcall me\b", r"\brepresentative\b"],
}

def matches_bucket(text, patterns):
    text_low = text.lower()
    return any(re.search(p, text_low) for p in patterns)

bucket_counts = {}
matched_any = 0

for name, patterns in candidate_buckets.items():
    count = df_msgs["text"].apply(lambda t: matches_bucket(t, patterns)).sum()
    bucket_counts[name] = count

print("Candidate bucket coverage (a message can match multiple buckets):")
for name, count in sorted(bucket_counts.items(), key=lambda x: -x[1]):
    pct = 100 * count / len(df_msgs)
    print(f"  {name}: {count} ({pct:.1f}%)")

# Kitne messages KISI bhi bucket me nahi aaye (taxonomy gap check)
def matches_any_bucket(text):
    return any(matches_bucket(text, patterns) for patterns in candidate_buckets.values())

unmatched = df_msgs[~df_msgs["text"].apply(matches_any_bucket)]
print(f"\nMessages matching NO bucket: {len(unmatched)} ({100*len(unmatched)/len(df_msgs):.1f}%)")

Candidate bucket coverage (a message can match multiple buckets):
  delivery_status: 22663 (37.4%)
  order_issue: 14579 (24.0%)
  prime_membership: 8021 (13.2%)
  refund_return: 4888 (8.1%)
  payment_billing: 4564 (7.5%)
  customer_service_complaint: 4563 (7.5%)
  account_access: 3447 (5.7%)
  product_issue: 1700 (2.8%)
  human_request: 463 (0.8%)

Messages matching NO bucket: 18754 (30.9%)


In [6]:
import random

random.seed(42)
sample_unmatched = unmatched.sample(25, random_state=42)

for i, row in sample_unmatched.iterrows():
    print(f"- {row['text']}")

- I had purchasedH6xon 111thJuly it showed battery prob on sep,submiited my mobile on BBSR SrvCentr 25thSept notyet rcvd
- Looks like a phishing scam
- Want to win a Fire TV Stick? Just watch any 5 titles- movies/TV shows! 10 winners a week till 31st Oct. #SuperDiwali
- At least leave it at the door or something instead of where anyone could just reach over and grab it.
- both my echos are no longer responding to voice commands.. bad update?? Or do I need an update??
- Is it possible to watch purchased films in different languages? We have a multilingual family.
- You guys made me to wait for such a long time, assured me of cashback. Is this the kind of service that you provide
- it’d be cool if we had the option to have alarms or timers apply to all the Echo devices in an audio group. If I set a timer on a dot in the basement it’d be nice to be able to check it in the kitchen.
- When will this become available ?
- and since then no information. No texts or calls on it's status. No det

In [7]:
candidate_buckets_v2 = {
    "delivery_status": [r"\bdeliver", r"\bpackage\b", r"\btrack", r"\barriv", r"\bshipp", r"\bwhere is\b", r"\breceived?\b", r"\brcvd\b"],
    "refund_return": [r"\brefund", r"\breturn", r"\bmoney back\b", r"\bexchange\b", r"\bcashback\b"],
    "order_issue": [r"\border\b", r"\bordered\b", r"\bcancel", r"\bwrong item\b"],
    "account_access": [r"\baccount\b", r"\bpassword\b", r"\blogin\b", r"\block", r"\baccess\b"],
    "payment_billing": [r"\bcharge", r"\bpayment\b", r"\bbill", r"\bcard\b"],
    "pricing_query": [r"\bmrp\b", r"\bprice\b", r"\bpricing\b", r"\boverchar", r"\bcost\b"],
    "prime_membership": [r"\bprime\b", r"\bmembership\b", r"\bsubscription\b"],
    "product_issue": [r"\bdamaged\b", r"\bbroken\b", r"\bdefective\b", r"\bnot working\b", r"\bfaulty\b"],
    "device_tech_support": [r"\becho\b", r"\balexa\b", r"\bkindle\b", r"\bwifi\b", r"\bapp\b", r"\bdevice\b", r"\bconnect", r"\bupdate\b"],
    "repair_service_status": [r"\brepair\b", r"\bservice cent", r"\bsubmitted\b.*\b(mobile|device|phone)\b"],
    "availability_question": [r"\bwhen (will|would)\b", r"\bavailable\b", r"\bavailability\b"],
    "feature_request": [r"\bwould be (cool|nice|great)\b", r"\bit'?d be (cool|nice)\b", r"\bsuggestion\b", r"\bfeature\b"],
    "customer_service_complaint": [r"\bcustomer service\b", r"\brude\b", r"\bworst\b", r"\bterrible\b", r"\bdisappointed\b", r"\bhorrible\b"],
    "human_request": [r"\bhuman\b", r"\breal person\b", r"\bspeak to\b", r"\bcall me\b", r"\brepresentative\b"],
}

def matches_bucket(text, patterns):
    text_low = text.lower()
    return any(re.search(p, text_low) for p in patterns)

bucket_counts_v2 = {}
for name, patterns in candidate_buckets_v2.items():
    count = df_msgs["text"].apply(lambda t: matches_bucket(t, patterns)).sum()
    bucket_counts_v2[name] = count

print("Updated bucket coverage:")
for name, count in sorted(bucket_counts_v2.items(), key=lambda x: -x[1]):
    pct = 100 * count / len(df_msgs)
    print(f"  {name}: {count} ({pct:.1f}%)")

def matches_any_v2(text):
    return any(matches_bucket(text, patterns) for patterns in candidate_buckets_v2.values())

unmatched_v2 = df_msgs[~df_msgs["text"].apply(matches_any_v2)]
print(f"\nStill unmatched: {len(unmatched_v2)} ({100*len(unmatched_v2)/len(df_msgs):.1f}%)")

Updated bucket coverage:
  delivery_status: 24697 (40.7%)
  order_issue: 14579 (24.0%)
  prime_membership: 8021 (13.2%)
  refund_return: 5275 (8.7%)
  customer_service_complaint: 4732 (7.8%)
  device_tech_support: 4648 (7.7%)
  account_access: 3447 (5.7%)
  payment_billing: 3178 (5.2%)
  product_issue: 1700 (2.8%)
  availability_question: 1134 (1.9%)
  pricing_query: 1013 (1.7%)
  human_request: 463 (0.8%)
  feature_request: 217 (0.4%)
  repair_service_status: 66 (0.1%)

Still unmatched: 14601 (24.1%)


In [8]:
sample_unmatched_v2 = unmatched_v2.sample(25, random_state=7)
for i, row in sample_unmatched_v2.iterrows():
    print(f"- {row['text']}")

- Just finished finale. What a wonderful, well-written and poignant show. Binge all 4 series on
- Recvd a sondmagic bullshit headphine instead a sony BT hdph. Your team is nt rdy 2 hlp. pls teach them quality cntrl
- I have an bought from the US. Despite releasing in India now, I'm still unable to change geo settings. Please help!
- it says audible library is empty even though books are there. How to resolve this?
- Will you or your company ever respond to my cry for help? or keep ignoring me?
- was this close to dispose off the packaging of my book till this dropped from it 🙂 thank you
- can I DM you instead?
- Pathetic Service. DO not use these cowboys. Take your money and do not provide. Crooks!!!
- Pretty upset right now with Trying to sleep on it so I don't call them right now and flip! #HorribleExperience
- Patanjali Special Chyawanprash with Saffron , 1 Kg #Loot #AmazonObhijaan #AmazonIndia
- Seriously #packagingoverkill
- #Congrats 2 of your chat agents managed to waste 2.5hrs 

In [9]:
candidate_buckets_v3 = dict(candidate_buckets_v2)  # pehle wale sab rakho
candidate_buckets_v3.update({
    "content_streaming_issue": [r"\bprime video\b", r"\baudible\b", r"\bsubtitle", r"\bepisode\b", r"\bgeo setting", r"\bregion\b", r"\bstream"],
    "promo_discount_query": [r"\bpromo code\b", r"\bdiscount\b", r"\bcoupon\b", r"\boffer\b", r"\bcashback\b"],
    "packaging_feedback": [r"\bpackaging\b", r"\bpackage overkill\b"],
})

def matches_any_v3(text):
    return any(matches_bucket(text, patterns) for patterns in candidate_buckets_v3.values())

unmatched_v3 = df_msgs[~df_msgs["text"].apply(matches_any_v3)]
print(f"Unmatched after v3: {len(unmatched_v3)} ({100*len(unmatched_v3)/len(df_msgs):.1f}%)")

Unmatched after v3: 13683 (22.6%)


In [10]:
# Heuristic: agar message me 2+ hashtags hai AUR koi complaint/question word nahi hai, likely promo/noise hai
def looks_promotional(text):
    hashtags = len(re.findall(r"#\w+", text))
    has_question_or_complaint = bool(re.search(r"\?|please|help|issue|problem|not|refund|cancel", text.lower()))
    return hashtags >= 2 and not has_question_or_complaint

promo_like = unmatched_v3[unmatched_v3["text"].apply(looks_promotional)]
print(f"Unmatched-and-promo-like: {len(promo_like)} ({100*len(promo_like)/len(df_msgs):.1f}% of total)")
print(f"Unmatched-and-genuine: {len(unmatched_v3) - len(promo_like)} ({100*(len(unmatched_v3)-len(promo_like))/len(df_msgs):.1f}% of total)")

Unmatched-and-promo-like: 475 (0.8% of total)
Unmatched-and-genuine: 13208 (21.8% of total)
